# Importing libraries

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from tqdm import tqdm

import imageio.v2 as imageio

import netCDF4 as nc
from netCDF4 import num2date, date2num, date2index
import pytz
import datetime

In [ ]:
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
plt.style.use("math.mplstyle")

# Loading dataset

In [ ]:
os.chdir("../edge-detection")
cwd = os.getcwd()
df = nc.Dataset(os.path.join(cwd, "landsat.nc"), "r")

In [ ]:
df

In [ ]:
print(np.diff(df["xdim"][:])[:5])
print(np.diff(df["ydim"][:])[:5])

In [ ]:
df["crs"]

In [ ]:
import numpy as np

n_scenes = df["SR_B5"].shape[0]
k = min(10, n_scenes)
rng = np.random.default_rng(42)
idx = rng.choice(n_scenes, size=k, replace=False)
print("Chosen indices:", idx)

img = np.ma.mean(df["SR_B5"][idx], axis=0)

In [ ]:
img.shape, img.dtype, img.min(), img.max()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import rotate
from skimage import measure


def extract_and_align_nonmasked_region(masked_array):
    binary_mask = masked_array.mask

    valid_mask = ~binary_mask

    labeled_mask, num_features = measure.label(valid_mask, return_num=True)

    regions = measure.regionprops(labeled_mask)

    regions.sort(key=lambda x: x.area, reverse=True)

    largest_region = regions[0]

    angle_rad = largest_region.orientation
    angle_degrees = np.degrees(angle_rad)

    rotation_angle = -angle_degrees

    min_row, min_col, max_row, max_col = largest_region.bbox

    extracted_data = masked_array.data[min_row:max_row, min_col:max_col]
    extracted_mask = masked_array.mask[min_row:max_row, min_col:max_col]

    extracted_masked_array = np.ma.array(extracted_data, mask=extracted_mask)

    aligned_data = rotate(
        extracted_data,
        rotation_angle,
        resize=True,
        preserve_range=True,
        mode="constant",
        cval=0,
    )

    aligned_mask = (
        rotate(
            extracted_mask.astype(float),
            rotation_angle,
            resize=True,
            preserve_range=True,
            mode="constant",
            cval=1,
        )
        > 0.5
    )

    aligned_image = np.ma.array(aligned_data, mask=aligned_mask)

    aligned_image = aligned_image.filled(0).astype(masked_array.dtype)

    print(f"Extracted region from [{min_row}:{max_row}, {min_col}:{max_col}]")
    print(f"Rotation angle applied: {rotation_angle:.2f} degrees")

    if np.any(aligned_mask):
        rows, cols = np.where(~aligned_mask)
        if len(rows) > 0 and len(cols) > 0:
            min_row, max_row = rows.min(), rows.max()
            min_col, max_col = cols.min(), cols.max()
            aligned_image = aligned_data[min_row : max_row + 1, min_col : max_col + 1]
            print(f"Cropped to final size: {aligned_image.shape}")

    return aligned_image


def display_fullscreen(image_data, cmap=matplotlib.cm.coolwarm):
    plt.imshow(image_data, cmap=cmap)
    plt.axis("off")
    plt.tight_layout(pad=0)
    plt.savefig("landsat.png", dpi=200, bbox_inches="tight", pad_inches=0)
    plt.show()

In [ ]:
aligned_image = extract_and_align_nonmasked_region(img)
display_fullscreen(aligned_image)

In [ ]:
img = aligned_image

In [ ]:
img.shape

In [ ]:
display_fullscreen(img[2400:2600, 2200:2400])

In [ ]:
img = img[2400:2600, 2200:2400]

In [ ]:
import numpy as np

if img.dtype == np.float64:
    if img.max() <= 1.0:
        img_uint8 = (img * 255).astype(np.uint8)
    else:
        img_uint8 = np.clip(img, 0, 255).astype(np.uint8)
else:
    img_uint8 = img.astype(np.uint8)

img = img_uint8

In [ ]:
os.chdir("../edge-detection")
os.getcwd()

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
from samgeo import SamGeo

transform = from_origin(0, 200, 1, 1)
with rasterio.open(
    "input.tif",
    "w",
    driver="GTiff",
    height=img.shape[0],
    width=img.shape[1],
    count=1,
    dtype=img.dtype,
    crs="EPSG:4326",
    transform=transform,
) as dst:
    dst.write(img, 1)

sam = SamGeo(
    model_type="vit_h",
    checkpoint="sam_vit_h.pth",
    automatic=True,
)


sam.generate(
    source="input.tif",
    output="masks.tif",
    min_size=20,
)

print(
    "Segmentation complete. Outputs: masks.tif (raster) and masks.geojson (vector polygons)."
)

In [ ]:
import matplotlib.pyplot as plt
import rasterio

with rasterio.open("input.tif") as src:
    img = src.read(1)

with rasterio.open("masks.tif") as src:
    mask = src.read(1)

plt.imshow(mask, cmap="tab20")
plt.axis("off")

plt.tight_layout()
plt.savefig("landsat_segmentation.png", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
import rasterio
import geopandas as gpd
from shapely.geometry import shape
from rasterio import features

with rasterio.open("masks.tif") as src:
    mask = src.read(1)
    transform = src.transform
    crs = src.crs

geoms = []
vals = []
for geom, val in features.shapes(mask, mask > 0, transform=transform):
    geoms.append(shape(geom))
    vals.append(val)

gdf = gpd.GeoDataFrame({"value": vals}, geometry=geoms, crs=crs)
gdf.to_file("masks.geojson", driver="GeoJSON")
print(f"Saved {len(gdf)} polygons to masks.geojson")

In [ ]:
import geopandas as gpd

min_area_threshold = 100

gdf = gpd.read_file("masks.geojson")

gdf["area"] = gdf.geometry.area
gdf = gdf[gdf["area"] >= min_area_threshold]

gdf = gdf.sort_values("area")

cleaned = []
for i, poly in gdf.iterrows():
    geom = poly.geometry
    new_cleaned = []
    for kept in cleaned:
        if kept.intersects(geom):
            kept = kept.difference(geom)
        if not kept.is_empty:
            new_cleaned.append(kept)
    cleaned = new_cleaned
    if not geom.is_empty:
        cleaned.append(geom)

cleaned_gdf = gpd.GeoDataFrame(geometry=cleaned, crs=gdf.crs)

cleaned_gdf.to_file("masks_cleaned.geojson", driver="GeoJSON")
print(f"Saved {len(cleaned_gdf)} polygons after cleaning")

In [ ]:
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd

with rasterio.open("input.tif") as src:
    img = src.read(1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

gdf = gpd.read_file("masks_cleaned.geojson")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img, cmap="gray", extent=extent, origin="upper", aspect="equal")

for geom in gdf.geometry:
    if geom.is_empty:
        continue
    x, y = geom.exterior.xy
    ax.plot(x, y, color="red", linewidth=1)

ax.set_title("Trimmed Segmentation Overlay")
ax.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
import numpy as np

with rasterio.open("input.tif") as src:
    img = src.read(1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

gdf = gpd.read_file("masks_cleaned.geojson")

patches = []
for geom in gdf.geometry:
    if geom.is_empty:
        continue
    x, y = geom.exterior.xy
    polygon = Polygon(np.column_stack([x, y]))
    patches.append(polygon)

num_polygons = len(patches)
colors = plt.cm.viridis(np.linspace(0, 1, num_polygons))

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img, cmap="gray", extent=extent, origin="upper", aspect="equal")

collection = PatchCollection(
    patches, facecolor=colors, edgecolor="black", linewidth=1, alpha=0.5
)
ax.add_collection(collection)

ax.axis("off")
plt.savefig("landsat-trimmed-segmentation.png", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()